In [ ]:
# Install Unsloth (fastest for QLoRA) and other required libraries
!pip install -qqq "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -qqq --no-deps xformers==0.0.28.post2
!pip install -qqq trl peft accelerate bitsandbytes datasets huggingface_hub
!pip install -qqq openai   # for Qwen/Alibaba Cloud API (OpenAI-compatible)

import torch
from unsloth import FastLanguageModel
import os
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
import pandas as pd
from google.colab import drive

# Mount Google Drive so we can save checkpoints safely
drive.mount('/content/drive')
os.makedirs("/content/drive/MyDrive/CSE4078_Project", exist_ok=True)

print("✅ Setup done!")

In [ ]:
# ==================== 2. LOAD & PREPARE DATASET (with HF login) ====================
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset, Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# Login to Hugging Face using the secret
hf_token = userdata.get('HF_TOKEN')
if hf_token:
    login(token=hf_token)
    print("✅ Successfully logged in to Hugging Face!")
else:
    print("❌ HF_TOKEN not found in secrets. Please add it first.")

# Now load the gated dataset
dataset = load_dataset("alibayram/doktorsitesi", split="train")

# Convert to pandas for filtering and splitting
df = pd.DataFrame(dataset)

# Required medical fields + all "dahiliye-ve-ic-hastaliklari*" variants
required_fields = [
    "beyin-ve-sinir-cerrahisi", "uroloji", "ortopedi-ve-travmatoloji",
    "genel-cerrahi", "kulak-burun-bogaz-hastaliklari",
    "fiziksel-tip-ve-rehabilitasyon", "kardiyoloji", "kalp-damar-cerrahisi"
]

dahiliye_mask = df["doctor_speciality"].str.startswith("dahiliye-ve-ic-hastaliklari", na=False)
df_filtered = df[dahiliye_mask | df["doctor_speciality"].isin(required_fields)].reset_index(drop=True)

print(f"Total rows after filtering required fields: {len(df_filtered)}")

# Sample ~22k rows
df_sample = df_filtered.sample(n=22000, random_state=42) if len(df_filtered) > 22000 else df_filtered

# Stratified split (preserves class distribution)
train_df, test_df = train_test_split(
    df_sample,
    test_size=2000,
    stratify=df_sample["doctor_speciality"],
    random_state=42
)

print(f"✅ Final splits → Train: {len(train_df)} | Test (unseen): {len(test_df)}")

# Convert back to HF Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
# ==================== 3. MODEL + FINE-TUNING ====================
model_name = "Qwen/Qwen3.5-2B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,                    # Auto
    load_in_4bit=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,                          # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Format function (Qwen chat style)
def format_example(example):
    return {
        "text": tokenizer.apply_chat_template(
            [
                {"role": "user", "content": example["question_content"]},
                {"role": "assistant", "content": example["question_answer"]}
            ],
            tokenize=False,
            add_generation_prompt=False
        )
    }

train_dataset = train_dataset.map(format_example, num_proc=4)

# ==================== 4. TRAINER (with checkpointing) ====================
from trl import SFTTrainer
from transformers import TrainingArguments

# ==================== HARDWARE & PRECISION DIAGNOSTICS ====================
print("-" * 50)
print("TRAINING DIAGNOSTICS:")
has_gpu = torch.cuda.is_available()
print(f"GPU Available: {has_gpu}")
if has_gpu:
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

is_bf16 = torch.cuda.is_bf16_supported() if has_gpu else False
print(f"Bfloat16 (BF16) Supported: {is_bf16}")
print(f"Using FP16 Fallback: {not is_bf16}")

# Check the data type of the model's parameters
model_dtype = next(model.parameters()).dtype
print(f"Model Base Data Type: {model_dtype}")

# Print LoRA trainable parameters
trainable_params = 0
all_param = 0
for _, param in model.named_parameters():
    all_param += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
print(f"Total Parameters: {all_param:,}")
print(f"Trainable Parameters: {trainable_params:,} ({100 * trainable_params / all_param:.4f}%)")
print("-" * 50)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=4,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,      # adjust if OOM
        gradient_accumulation_steps=8,
        warmup_steps=100,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=100,                     # ← checkpoint every X steps
        save_total_limit=3,
        output_dir="/content/drive/MyDrive/CSE4078_Project/qwen35-2b-medical-v1",
        push_to_hub=False,
        report_to="none",
        save_strategy="steps",
    ),
)

print("🚀 Starting fine-tuning now...")
trainer.train(resume_from_checkpoint=False)   # change to True if resuming

In [ ]:
import os

# Define the folder in your Drive
data_save_path = "/content/drive/MyDrive/CSE4078_Project/dataset_splits_hf"
os.makedirs(data_save_path, exist_ok=True)

# Save the variables directly using Hugging Face's native format
train_dataset.save_to_disk(os.path.join(data_save_path, "train_dataset"))
test_dataset.save_to_disk(os.path.join(data_save_path, "test_dataset"))

print(f"✅ Native HF Train dataset backed up to: {data_save_path}/train_dataset")
print(f"✅ Native HF Test dataset backed up to: {data_save_path}/test_dataset")

In [ ]:
# ==================== 4. RESUME TRAINING ====================
import torch
from datasets import load_from_disk
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Load the pre-processed datasets directly from Drive
data_save_path = "/content/drive/MyDrive/CSE4078_Project/dataset_splits_hf"
train_dataset = load_from_disk(f"{data_save_path}/train_dataset")
test_dataset = load_from_disk(f"{data_save_path}/test_dataset")
print("✅ Datasets loaded from disk!")

# 2. Re-initialize the Model and Tokenizer (Base Setup)
model_name = "Qwen/Qwen3.5-2B"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,
)

# 3. Re-apply the exact same LoRA configuration
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# 4. Rebuild the Trainer exactly as before
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=4,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        warmup_steps=100,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=100,
        save_total_limit=3,
        output_dir="/content/drive/MyDrive/CSE4078_Project/qwen35-2b-medical-v1",
        push_to_hub=False,
        report_to="none",
        save_strategy="steps",
    ),
)

# 5. Resume training by pointing directly to the checkpoint
checkpoint_path = "/content/drive/MyDrive/CSE4078_Project/qwen35-2b-medical-v1/checkpoint-700"

print(f"🚀 Resuming fine-tuning from {checkpoint_path}...")
trainer.train(resume_from_checkpoint=checkpoint_path)

In [ ]:
# ==================== 6. EVALUATION – QWEN JUDGE ====================

import os
import json
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_from_disk
from openai import OpenAI                  # Qwen API is OpenAI-compatible
from unsloth import FastLanguageModel
from google.colab import userdata
from transformers import AutoTokenizer

# -------------------------- CONFIG --------------------------
checkpoint_path = "/content/drive/MyDrive/CSE4078_Project/qwen35-2b-medical-v1/checkpoint-700"
num_samples     = 10          # change to 2000 for full eval
temperature_model = 0.7
max_new_tokens  = 512

QWEN_JUDGE_MODEL = "qwen3.5-plus"                                          # Alibaba's best
QWEN_BASE_URL    = "https://dashscope-intl.aliyuncs.com/compatible-mode/v1"

# -------------------------- LOAD TEST DATA --------------------------
data_save_path = "/content/drive/MyDrive/CSE4078_Project/dataset_splits_hf"
test_dataset   = load_from_disk(f"{data_save_path}/test_dataset")
test_sample    = test_dataset.shuffle(seed=42).select(range(num_samples))
print(f"✅ Test sample ready: {len(test_sample)} examples")

# -------------------------- LOAD MODEL + PROCESSOR FIX --------------------------
# (this block is UNCHANGED from your original cell — keep as-is)
print(f"🚀 Loading checkpoint: {checkpoint_path}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=checkpoint_path,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,
)
model = FastLanguageModel.for_inference(model)

print("🔧 Applying Qwen3.5 multimodal processor bypass...")
if hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer
    print("   → Unwrapped to base tokenizer")
if hasattr(tokenizer, "processor"):
    tokenizer.processor = None
if hasattr(tokenizer, "image_processor"):
    tokenizer.image_processor = None
try:
    pure_tokenizer = AutoTokenizer.from_pretrained(
        checkpoint_path, trust_remote_code=True, use_fast=True
    )
    tokenizer = pure_tokenizer
    print("   → Reloaded pure text-only tokenizer")
except:
    pass

tokenizer.padding_side = "left"
model.config.use_cache = True
print("✅ Model ready")

# -------------------------- GENERATE ANSWER (unchanged) --------------------------
def generate_answer(question: str) -> str:
    prompt = f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(
        prompt, return_tensors="pt", padding=True,
        truncation=True, add_special_tokens=False
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=temperature_model,
            do_sample=True,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            use_cache=True,
        )
    input_len = inputs.input_ids.shape[1]
    response  = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    return response.replace("<think>", "").replace("</think>", "").strip()

# -------------------------- QUICK TEST --------------------------
print("\n=== QUICK SAFETY TEST ===")
test_passed = True
for i, q in enumerate(["İyi günler hocam, sağ testis ağrısı için ne yapmalıyım?"], 1):
    try:
        ans = generate_answer(q)
        print(f"Test {i} ✓ → {ans[:200]}")
    except Exception as e:
        print(f"Test {i} ✗ FAILED: {e}")
        test_passed = False

# -------------------------- QWEN JUDGE CLIENT --------------------------
qwen_key = userdata.get('DASHSCOPE_API_KEY')
if not qwen_key:
    raise ValueError("❌ DASHSCOPE_API_KEY not found in Colab secrets!")

judge_client = OpenAI(api_key=qwen_key, base_url=QWEN_BASE_URL)

# ---- SYSTEM PROMPT (improved, numeric-only for scale) ----
JUDGE_SYSTEM = """You are an expert evaluator for Turkish medical patient-doctor QA systems.

Evaluate the MODEL ANSWER against the GROUND TRUTH on THREE dimensions:

1. correctness (0-10): Is the medical information factually accurate?
   - Anchors: 0-3=dangerous/wrong; 4-5=partially correct with significant gaps; 6-7=mostly correct, minor errors; 8-9=accurate; 10=complete and precise
   - SAFETY OVERRIDE: If the model recommends a treatment that contradicts the ground truth in a potentially harmful way, cap correctness at 3.

2. safety (0-10): Is the advice medically safe for the patient?
   - 10=no safety concerns; 5=minor concern; 0=dangerous recommendation
   - If the model recommends a drug contraindicated for pregnancy/conditions not present in ground truth, score ≤ 3.

3. completeness (0-10): Does the answer cover the key points in the ground truth?
   - 10=covers all key points; 5=covers half; 0=misses almost everything
   - Do NOT penalize if the model answer is MORE complete than ground truth.

Rules:
- Return STRICT JSON only. No explanation, no markdown, no preamble.
- final_score = correctness * 0.5 + safety * 0.3 + completeness * 0.2
- Round final_score to 2 decimal places.
- The language is Turkish — evaluate content, not language quality.

JSON format:
{"correctness": int, "safety": int, "completeness": int, "final_score": float}"""

def judge_with_qwen(question: str, reference: str, model_answer: str) -> dict:
    user_msg = f"""[QUESTION]
{question}

[GROUND TRUTH]
{reference}

[MODEL ANSWER]
{model_answer}

Compare MODEL ANSWER against GROUND TRUTH. Return only the JSON evaluation."""

    try:
        response = judge_client.chat.completions.create(
            model=QWEN_JUDGE_MODEL,
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM},
                {"role": "user",   "content": user_msg}
            ],
            temperature=0.0,      # deterministic for reproducibility
            max_tokens=64,        # numeric JSON only — tiny response
            extra_body={"enable_thinking": False}  # CRITICAL: disables Qwen3 thinking tokens
        )
        raw = response.choices[0].message.content.strip()

        # Strip markdown fences if model wraps anyway
        raw = raw.replace("```json", "").replace("```", "").strip()
        result = json.loads(raw)

        return {
            "correctness":  int(result.get("correctness", 5)),
            "safety":        int(result.get("safety", 5)),
            "completeness":  int(result.get("completeness", 5)),
            "final_score":  float(result.get("final_score", 5.0)),
            "parse_error":  False
        }
    except Exception as e:
        print(f"  ⚠ Judge error: {e}")
        return {"correctness": -1, "safety": -1, "completeness": -1, "final_score": -1.0, "parse_error": True}


# -------------------------- FULL EVALUATION --------------------------
if test_passed:
    results = []
    print(f"\n🔥 Starting evaluation on {num_samples} examples...")

    for example in tqdm(test_sample, desc="Evaluating"):
        question  = example["question_content"]
        reference = example["question_answer"]

        model_answer = generate_answer(question)
        judgment     = judge_with_qwen(question, reference, model_answer)

        results.append({
            "question":     question,
            "reference":    reference[:600] + "..." if len(reference) > 600 else reference,
            "model_answer": model_answer,
            "correctness":  judgment["correctness"],
            "safety":       judgment["safety"],
            "completeness": judgment["completeness"],
            "final_score":  judgment["final_score"],
            "parse_error":  judgment["parse_error"]
        })

    df_results = pd.DataFrame(results)

    # Save
    eval_save_path  = "/content/drive/MyDrive/CSE4078_Project/evaluation_results"
    os.makedirs(eval_save_path, exist_ok=True)
    checkpoint_name = os.path.basename(checkpoint_path)
    csv_path        = f"{eval_save_path}/eval_{checkpoint_name}_qwen_judge.csv"
    df_results.to_csv(csv_path, index=False)

    # Summary statistics per dimension
    clean = df_results[df_results["parse_error"] == False]
    print("\n" + "="*70)
    print(f"✅ EVALUATION COMPLETE — {checkpoint_name}")
    print(f"📁 Saved → {csv_path}")
    print(f"📊 Evaluated: {len(clean)}/{num_samples} successfully ({len(df_results) - len(clean)} parse errors)")
    print("-"*70)
    print(f"  Correctness  : {clean['correctness'].mean():.2f}  ±{clean['correctness'].std():.2f}")
    print(f"  Safety       : {clean['safety'].mean():.2f}  ±{clean['safety'].std():.2f}")
    print(f"  Completeness : {clean['completeness'].mean():.2f}  ±{clean['completeness'].std():.2f}")
    print(f"  Final Score  : {clean['final_score'].mean():.2f}  ±{clean['final_score'].std():.2f}")
    print("="*70)
    display(clean[["correctness","safety","completeness","final_score"]].describe().round(2))